# Saudi Arabia Hourly Demand Forecasting — XGBoost

## Goal
Train and evaluate an XGBoost regression model to predict `demand_count` for each
`(city, region_id, aoi_id, bucket_hour)` combination.

**Dataset:** Synthetic Saudi demand dataset generated by `saudi_demand_generator_final.py`.
Calibrated to TGA 2024 (290M national orders, Eastern Province 15%, Dammam ~71K orders/day).

**Feature design principle:** The generator uses hidden mechanics (exact multiplier curves,
prayer-dip magnitudes, hour-specific Ramadan reshaping) that are NOT in the CSV. The CSV
contains only binary calendar flags and temporal indices. The model must learn the shape
and magnitude of effects from historical demand patterns, not from the generator's lookup tables.

**Event flags in CSV (from generator, not recomputed here):**
`is_ramadan`, `is_eid`, `is_eid_pre`, `is_white_friday`, `is_national_day`,
`is_founding_day`, `is_back_to_school`, `is_hajj`, `is_summer`, `is_weekend`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import os
warnings.filterwarnings('ignore')

try:
    import xgboost as xgb
    print("XGBoost version:", xgb.__version__)
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "xgboost", "-q"])
    import xgboost as xgb
    print("XGBoost installed, version:", xgb.__version__)

SEED = 42
np.random.seed(SEED)

# Tree method — change to 'gpu_hist' if you have GPU XGBoost build
TREE_METHOD = "hist"

print(f"tree_method: '{TREE_METHOD}'")
print("Ready.")

## Step 1 — Load Dataset (always fresh from disk)

**No stale DataFrame reuse.** Every run loads from CSV explicitly.
The CSV path must match what `saudi_demand_generator_final.py` produces.

In [ ]:
# ── Single source of truth for file path ──────────────────────────────────────
# LOCAL: set this to the directory containing saudi_hourly_features.csv
PROJECT_DIR = os.path.dirname(os.path.abspath("__file__"))
CSV_PATH = os.path.join(PROJECT_DIR, "saudi_hourly_features.csv")

# COLAB: uncomment these two lines instead
# from google.colab import drive
# drive.mount('/content/drive')
# CSV_PATH = "/content/drive/MyDrive/Senior-Project-Model/Saudi-model/saudi_hourly_features.csv"

# ── Always fresh load ─────────────────────────────────────────────────────────
print(f"Loading from: {CSV_PATH}")
model_df = pd.read_csv(CSV_PATH, parse_dates=["bucket_hour"])
print(f"Loaded shape: {model_df.shape}")
print(f"Columns: {list(model_df.columns)}")

## Step 1a — Data Quality Checks

Verify the CSV before doing anything else. Print row counts, date range,
zero-demand share, per-AOI variability, and cross-AOI correlation.

In [ ]:
print("=" * 60)
print("  DATA QUALITY CHECKS")
print("=" * 60)

# Basic counts
print(f"\nTotal rows:      {len(model_df):,}")
print(f"Unique AOIs:     {model_df['aoi_id'].nunique()}")
print(f"Unique hours:    {model_df['bucket_hour'].nunique():,}")
print(f"Date range:      {model_df['bucket_hour'].min()} to {model_df['bucket_hour'].max()}")

# Zero-demand share
zero_share = (model_df["demand_count"] == 0).mean()
print(f"\nZero-demand share: {zero_share:.4%}")
if zero_share < 0.05:
    print("  -> Less than 5%: zero-inflated modeling is NOT warranted.")
else:
    print(f"  -> {zero_share:.1%} zeros: consider zero-inflated approach.")

# Per-AOI variability
print("\nPer-AOI demand statistics:")
aoi_stats = model_df.groupby(["aoi_id", "aoi_name"])["demand_count"].agg(
    ["mean", "std", "min", "max"]
)
aoi_stats["cv"] = aoi_stats["std"] / aoi_stats["mean"]
print(aoi_stats.to_string())

# Cross-AOI correlation
print("\nCross-AOI hourly demand correlations:")
pivot = model_df.pivot_table(
    index="bucket_hour", columns="aoi_id", values="demand_count", aggfunc="first"
).dropna()
corr_matrix = pivot.corr()
mask = ~np.eye(corr_matrix.shape[0], dtype=bool)
print(f"  Mean pairwise: {corr_matrix.values[mask].mean():.4f}")
print(f"  Min: {corr_matrix.values[mask].min():.4f}  |  Max: {corr_matrix.values[mask].max():.4f}")
if corr_matrix.values[mask].mean() > 0.95:
    print("  WARNING: AOIs are nearly identical. Model has little spatial variation to learn.")

# Missing values in lag features
lag_cols = [c for c in model_df.columns if c.startswith("lag_") or c.startswith("roll_")]
if lag_cols:
    print(f"\nMissing values in lag/rolling features:")
    print(model_df[lag_cols].isnull().sum().to_string())
else:
    print("\nNo lag features found in CSV (they are pre-computed by generator).")

## Step 2 — Feature Selection

**No event flag recomputation.** The CSV already contains all event flags from the
generator. Using the CSV as single source of truth avoids any risk of date mismatches.

Features used by the model:
- **Temporal:** `hour`, `day_of_week`, `month` (raw indices — the model learns patterns)
- **Cyclical encoding:** `hour_sin`, `hour_cos` (helps with hour-of-day continuity)
- **Calendar flags:** `is_ramadan`, `is_eid`, `is_eid_pre`, `is_white_friday`,
  `is_national_day`, `is_founding_day`, `is_back_to_school`, `is_hajj`, `is_summer`, `is_weekend`
- **Lag features:** `lag_1`, `lag_2`, `lag_24`, `lag_48`, `lag_168`
- **Rolling features:** `roll_24_mean`, `roll_168_mean`
- **Spatial:** `region_id`, `aoi_id`

**What is NOT a feature:** The generator's internal hourly profile curves,
prayer-dip magnitudes, event multiplier values, summer suppression curves.
These are hidden mechanics that shaped demand_count but are not in the CSV.

In [ ]:
# ── Add cyclical hour encoding ────────────────────────────────────────────────
model_df["hour_sin"] = np.sin(2 * np.pi * model_df["hour"] / 24)
model_df["hour_cos"] = np.cos(2 * np.pi * model_df["hour"] / 24)

# ── Define feature columns ────────────────────────────────────────────────────
TARGET = "demand_count"

FEATURE_COLS = [
    # Temporal
    "hour", "day_of_week", "month",
    "hour_sin", "hour_cos",
    # Calendar flags (from CSV, not recomputed)
    "is_ramadan", "is_eid", "is_eid_pre", "is_white_friday",
    "is_national_day", "is_founding_day", "is_back_to_school",
    "is_hajj", "is_summer", "is_weekend",
    # Lag features
    "lag_1", "lag_2", "lag_24", "lag_48", "lag_168",
    # Rolling features
    "roll_24_mean", "roll_168_mean",
    # Spatial
    "region_id", "aoi_id",
]

# Verify all features exist
missing_features = [c for c in FEATURE_COLS if c not in model_df.columns]
if missing_features:
    raise ValueError(f"Missing features in CSV: {missing_features}")

# Drop rows with NaN in any feature (lag warmup rows)
valid_mask = model_df[FEATURE_COLS].notna().all(axis=1)
model_df = model_df[valid_mask].reset_index(drop=True)

print(f"Features: {len(FEATURE_COLS)}")
print(f"Rows after dropping NaN lag rows: {len(model_df):,}")
print(f"Feature list: {FEATURE_COLS}")

## Step 3 — Chronological Train / Val / Test Split

Split globally by time (60/20/20). All AOIs share the same time boundaries.

**Lag leakage note:** Lag features (e.g., `lag_168`) in the first validation rows
use actual demand from the last training hours. This is correct for a production
forecasting setup (at prediction time, you DO have last week's actuals). We verify
that no future data leaks backward.

In [ ]:
TRAIN_RATIO = 0.60
VAL_RATIO   = 0.20

all_hours = np.sort(model_df["bucket_hour"].unique())
n_hours   = len(all_hours)

train_end = all_hours[int(n_hours * TRAIN_RATIO) - 1]
val_end   = all_hours[int(n_hours * (TRAIN_RATIO + VAL_RATIO)) - 1]

train_mask = model_df["bucket_hour"] <= train_end
val_mask   = (model_df["bucket_hour"] > train_end) & (model_df["bucket_hour"] <= val_end)
test_mask  = model_df["bucket_hour"] > val_end

X_train = model_df.loc[train_mask, FEATURE_COLS]
y_train = model_df.loc[train_mask, TARGET]
X_val   = model_df.loc[val_mask, FEATURE_COLS]
y_val   = model_df.loc[val_mask, TARGET]
X_test  = model_df.loc[test_mask, FEATURE_COLS]
y_test  = model_df.loc[test_mask, TARGET]

print(f"Train: {len(X_train):>7,} rows  "
      f"({model_df.loc[train_mask, 'bucket_hour'].min().date()} to "
      f"{model_df.loc[train_mask, 'bucket_hour'].max().date()})")
print(f"Val:   {len(X_val):>7,} rows  "
      f"({model_df.loc[val_mask, 'bucket_hour'].min().date()} to "
      f"{model_df.loc[val_mask, 'bucket_hour'].max().date()})")
print(f"Test:  {len(X_test):>7,} rows  "
      f"({model_df.loc[test_mask, 'bucket_hour'].min().date()} to "
      f"{model_df.loc[test_mask, 'bucket_hour'].max().date()})")

# ── Verify no future leakage ──────────────────────────────────────────────────
train_max_hour = model_df.loc[train_mask, "bucket_hour"].max()
val_min_hour   = model_df.loc[val_mask, "bucket_hour"].min()
val_max_hour   = model_df.loc[val_mask, "bucket_hour"].max()
test_min_hour  = model_df.loc[test_mask, "bucket_hour"].min()

assert val_min_hour > train_max_hour, "Validation leaks into training!"
assert test_min_hour > val_max_hour, "Test leaks into validation!"
print("\nNo temporal leakage detected. Splits are clean.")

## Step 4 — Baseline Models

Five baselines to benchmark against:
- **lag_1**: predict = demand 1 hour ago
- **lag_24**: predict = demand 24 hours ago (same hour yesterday)
- **lag_168**: predict = demand 168 hours ago (same hour, same weekday last week) — **the baseline to beat** for hourly weekly-seasonal data
- **roll_24_mean**: 24-hour rolling average
- **roll_168_mean**: 7-day rolling average

In [ ]:
def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def smape(y_true, y_pred):
    denom = np.abs(y_true) + np.abs(y_pred)
    mask = denom > 0
    return np.mean(2 * np.abs(y_true[mask] - y_pred[mask]) / denom[mask]) * 100

def evaluate(y_true, y_pred, name, split_name):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return {
        "Model": name,
        "Split": split_name,
        "MAE":   round(mae(y_true, y_pred), 4),
        "RMSE":  round(rmse(y_true, y_pred), 4),
        "sMAPE": round(smape(y_true, y_pred), 2),
    }

results = []

BASELINES = {
    "Baseline lag_1":        "lag_1",
    "Baseline lag_24":       "lag_24",
    "Baseline lag_168":      "lag_168",
    "Baseline roll_24_mean": "roll_24_mean",
    "Baseline roll_168_mean":"roll_168_mean",
}

for split_name, X_s, y_s in [("val", X_val, y_val), ("test", X_test, y_test)]:
    for bname, col in BASELINES.items():
        results.append(evaluate(y_s, X_s[col], bname, split_name))

baseline_df = pd.DataFrame(results)
print("\n=== Baseline Results ===")
print(baseline_df.to_string(index=False))
print("\nlag_168 is the key baseline — XGBoost must beat it to justify model complexity.")

## Step 5 — Train XGBoost with Early Stopping

In [ ]:
model = xgb.XGBRegressor(
    objective          = "reg:squarederror",
    tree_method        = TREE_METHOD,
    n_estimators       = 3000,
    learning_rate      = 0.05,
    max_depth          = 6,
    min_child_weight   = 10,
    subsample          = 0.8,
    colsample_bytree   = 0.8,
    reg_alpha          = 0.1,
    reg_lambda         = 2.0,
    random_state       = SEED,
    n_jobs             = -1,
    eval_metric        = ["rmse", "mae"],
    early_stopping_rounds = 100,
)

model.fit(
    X_train, y_train,
    eval_set = [(X_train, y_train), (X_val, y_val)],
    verbose  = 100,
)

print(f"\nBest iteration: {model.best_iteration}")

## Step 6 — Training Curves

In [ ]:
def plot_train_val_curves(evals_result, title="Model", metric="rmse"):
    fig, ax = plt.subplots(figsize=(10, 5))
    if "validation_0" in evals_result:
        train_metric = evals_result["validation_0"][metric]
        ax.plot(range(1, len(train_metric) + 1), train_metric,
                label=f"Train {metric.upper()}", color="steelblue", linewidth=1.5)
    if "validation_1" in evals_result:
        val_metric = evals_result["validation_1"][metric]
        ax.plot(range(1, len(val_metric) + 1), val_metric,
                label=f"Val {metric.upper()}", color="tomato", linewidth=1.5)
    ax.set_xlabel("Boosting round")
    ax.set_ylabel(metric.upper())
    ax.set_title(f"{title} — Training vs Validation")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_train_val_curves(model.evals_result(), title="XGBoost (main)")

## Step 7 — Evaluate All Models

Compare XGBoost against all baselines. The model should beat lag_168 to be worth using.

In [ ]:
val_preds  = np.clip(model.predict(X_val), 0, None)
test_preds = np.clip(model.predict(X_test), 0, None)

results.append(evaluate(y_val,  val_preds,  "XGBoost", "val"))
results.append(evaluate(y_test, test_preds, "XGBoost", "test"))

results_df = pd.DataFrame(results)
comparison = results_df.pivot(index="Model", columns="Split", values=["MAE", "RMSE", "sMAPE"])
comparison.columns = [f"{metric}_{split}" for metric, split in comparison.columns]
comparison = comparison.reindex(
    columns=["MAE_val", "RMSE_val", "sMAPE_val", "MAE_test", "RMSE_test", "sMAPE_test"]
)

print("\n" + "=" * 80)
print("  MODEL COMPARISON")
print("=" * 80)
print(comparison.to_string())

# ── Explicit check: does XGBoost beat lag_168? ────────────────────────────────
xgb_test_mae = results_df[(results_df["Model"]=="XGBoost") & (results_df["Split"]=="test")]["MAE"].values[0]
lag168_test_mae = results_df[(results_df["Model"]=="Baseline lag_168") & (results_df["Split"]=="test")]["MAE"].values[0]

improvement = (lag168_test_mae - xgb_test_mae) / lag168_test_mae * 100
if improvement > 0:
    print(f"\nXGBoost beats lag_168 by {improvement:.1f}% MAE on test set.")
else:
    print(f"\nWARNING: XGBoost does NOT beat lag_168. Improvement: {improvement:.1f}%")
    print("The model may not be learning anything beyond weekly seasonality.")

## Step 8 — Feature Importance

In [ ]:
importance = (
    pd.Series(model.feature_importances_, index=FEATURE_COLS)
    .sort_values(ascending=False)
    .head(20)
)

fig, ax = plt.subplots(figsize=(10, 6))
importance.plot(kind="barh", ax=ax, color="steelblue")
ax.invert_yaxis()
ax.set_title("XGBoost — Top 20 Feature Importances (gain)")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()

# Sanity check: if event flags dominate, the task might still be too easy
event_flags = [c for c in FEATURE_COLS if c.startswith("is_")]
event_importance = importance.reindex(event_flags).dropna().sum()
total_importance = importance.sum()
print(f"\nEvent flags share of top-20 importance: {event_importance/total_importance:.1%}")
if event_importance / total_importance > 0.5:
    print("WARNING: Event flags dominate. Model may be over-relying on calendar features.")

## Step 9 — Example AOI Plot: Actual vs Predicted

In [ ]:
test_slice = model_df.loc[test_mask].copy()
test_slice["pred_xgb"]    = test_preds
test_slice["pred_lag24"]  = X_test["lag_24"].values
test_slice["pred_lag168"] = X_test["lag_168"].values

# Pick busiest AOI by total test demand
best_aoi = (
    test_slice.groupby(["city", "region_id", "aoi_id"])["demand_count"]
    .sum().idxmax()
)
city_s, region_s, aoi_s = best_aoi

aoi_test = (
    test_slice[
        (test_slice["city"] == city_s) &
        (test_slice["region_id"] == region_s) &
        (test_slice["aoi_id"] == aoi_s)
    ].sort_values("bucket_hour")
)

PLOT_DAYS = 14
aoi_plot = aoi_test.head(24 * PLOT_DAYS)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(aoi_plot["bucket_hour"], aoi_plot["demand_count"],
        label="Actual", color="black", linewidth=1.5)
ax.plot(aoi_plot["bucket_hour"], aoi_plot["pred_xgb"],
        label="XGBoost", color="steelblue", linewidth=1.2, linestyle="--")
ax.plot(aoi_plot["bucket_hour"], aoi_plot["pred_lag168"],
        label="Lag-168", color="tomato", linewidth=1.0, linestyle=":")

ax.set_title(
    f"Hourly Demand Forecast — AOI {aoi_s} (region {region_s}, {city_s})\n"
    f"Test set, first {PLOT_DAYS} days"
)
ax.set_xlabel("Time")
ax.set_ylabel("demand_count")
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print(f"Selected AOI: city={city_s}, region_id={region_s}, aoi_id={aoi_s}")
print(f"Test rows for this AOI: {len(aoi_test)}")

## Step 10 — Per-AOI Error Analysis

Check if the model performs consistently across zones or if some AOIs are poorly served.

In [ ]:
test_slice["abs_error_xgb"]   = np.abs(test_slice["demand_count"] - test_slice["pred_xgb"])
test_slice["abs_error_lag168"] = np.abs(test_slice["demand_count"] - test_slice["pred_lag168"])

per_aoi = test_slice.groupby(["aoi_id", "aoi_name"]).agg(
    mean_demand=("demand_count", "mean"),
    mae_xgb=("abs_error_xgb", "mean"),
    mae_lag168=("abs_error_lag168", "mean"),
).round(2)

per_aoi["xgb_vs_lag168_%"] = (
    (per_aoi["mae_lag168"] - per_aoi["mae_xgb"]) / per_aoi["mae_lag168"] * 100
).round(1)

print("\n=== Per-AOI Test Set Performance ===")
print(per_aoi.to_string())

## What This Synthetic Experiment Can and Cannot Prove

### What it CAN demonstrate:
- That the XGBoost pipeline (data loading, feature engineering, chronological splitting,
  training, evaluation) is mechanically correct and reproducible.
- That the model can learn non-trivial temporal patterns (hourly shape, weekly seasonality,
  event effects) from lag features and binary calendar flags alone, without being handed
  the generator's exact rules.
- That the model beats meaningful baselines (especially lag_168) on held-out future data.
- That the evaluation framework (MAE, RMSE, sMAPE, per-AOI breakdown) is sound.

### What it CANNOT prove:
- That this model will work on real delivery data. Synthetic data has known structure;
  real data has unknown structure, distribution shifts, and external shocks.
- That the event multipliers are correct. Most are assumptions within directional ranges.
- That AOI differentiation reflects real geographic variation. Zone parameters are invented.
- That the noise model (negative binomial + local shocks) matches real delivery variance.
- That absolute error magnitudes are realistic. Only relative comparisons (model vs baseline)
  are meaningful.
- That the model generalizes beyond the patterns baked into the generator. If the generator
  doesn't model a real-world phenomenon, the model cannot learn it.

### Bottom line:
This is a **pipeline validation exercise**, not a demand forecasting proof. It demonstrates
that the engineering works. Proving forecast quality requires real data.

## Final Verification Checklist

Run these checks before trusting results:

1. **Row count**: CSV should have ~85K+ rows (8784 hours x 10 AOIs, minus ~0.5% missing, minus lag warmup)
2. **Date range**: First hour = 2024-01-08+ (after 168h warmup), last hour = 2024-12-31 23:00
3. **Zero-demand share**: Should be small (< 2%) — if not, reconsider noise parameters
4. **Cross-AOI correlation**: Should be < 0.90 mean pairwise (zones are differentiated)
5. **Per-AOI CV**: Coefficient of variation should vary across zones (not all identical)
6. **No temporal leakage**: val_min_hour > train_max_hour, test_min_hour > val_max_hour
7. **XGBoost beats lag_168**: If not, the model adds no value over simple weekly repetition
8. **Feature importance**: Lag features should rank high; event flags should contribute but not dominate
9. **Training curve**: Val loss should plateau, not diverge (overfitting check)
10. **Per-AOI errors**: No single AOI should have wildly different error patterns
11. **Determinism**: Re-running the notebook produces identical numbers (fixed seeds)
12. **Path consistency**: CSV_PATH in notebook matches SAVE_PATH/CSV_FILENAME in generator